In [13]:
import pandas as pd 
import yaml
import sys

users_df = pd.read_csv("../data2/users.csv") 

with open("../data2/books.yaml") as books:
    data=yaml.safe_load(books) 
books_df = pd.DataFrame(data)

orders_df = pd.read_parquet("../data2/orders.parquet")

In [14]:
users_df["phone"] = users_df["phone"].str.replace(r"\D", "", regex=True)

In [15]:
users_df.head()

,id,name,address,phone,email
0,53386,Jacqulyn Mante,"566 Emelina Turnpike, Lashundaside, NE 78034-5281",9627158009,leonardo.leannon@wintheiser-lueilwitz.example
1,54635,Devon Leannon III,,5371339394,jon_torp@bayer.example
2,55435,Shiloh Keebler,"425 Wesley Hills, Nathanialburgh, MS 08004",4742423397,murray_mcclure@pfeffer.example
3,53627,Luke Rolfson,"Suite 556 20616 Little Union, Connieside, VT 3...",9273818818,murray.shanahan@heathcote.example
4,55650,Prince Braun,"981 Howell Spring, New Sheldon, LA 62246-1232",6861861306,anitra@pollich-kris.example


In [16]:
users_df["phone_length"] = users_df["phone"].str.len()
users_df["phone_length"].value_counts()

phone_length
10    2810
Name: count, dtype: int64

In [17]:
users_df = users_df.drop(columns=['phone_length'])

In [18]:
import numpy as np
users_df['address'] = users_df['address'].replace(r'^\s*$', np.nan, regex=True)

In [19]:
books_df.columns = books_df.columns.str.strip(':')
books_df.columns

Index(['id', 'title', 'author', 'genre', 'publisher', 'year'], dtype='object')

In [20]:
books_df['publisher'] = books_df['publisher'].replace(r'^\s*$', np.nan, regex=True)
books_df['year'] = books_df['year'].replace(r'^\s*$', np.nan, regex=True)

In [21]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 741 entries, 0 to 740
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         741 non-null    int64 
 1   title      741 non-null    object
 2   author     741 non-null    object
 3   genre      741 non-null    object
 4   publisher  726 non-null    object
 5   year       736 non-null    object
dtypes: int64(1), object(5)
memory usage: 34.9+ KB


In [22]:
books_df['year'] = pd.to_numeric(books_df['year'], errors='coerce').astype('Int64')

In [23]:
import re

patterns = [
    r'\d{4}-\d{2}-\d{2}',                   
    r'\d{2}/\d{2}/\d{2,4}',              
    r'\d{1,2}-[A-Za-z]+-\d{4}',              
    r'\d{1,2}-[A-Za-z]{3}-\d{4}',           
    r'\d{1,2}\.\d{1,2}\.\d{4}',             
    r'[A-Za-z]{3}\s[A-Za-z]{3}\s+\d{1,2}\s[\d:]+\s\d{4}',  
]

def extract_date(ts):
    if pd.isnull(ts):
        return None
    ts = str(ts)
    for pattern in patterns:
        match = re.search(pattern, ts)
        if match:
            return match.group()
    return None

orders_df['date'] = orders_df['timestamp'].apply(extract_date)
orders_df['date'] = pd.to_datetime(orders_df['date'], errors='coerce')
print(orders_df['date'].isnull().sum())

C:\Users\User\AppData\Local\Temp\ipykernel_41248\1336603486.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders_df['date'] = pd.to_datetime(orders_df['date'], errors='coerce')


0


In [24]:
orders_df.head()

,id,user_id,book_id,quantity,unit_price,timestamp,shipping,date
0,75198,55590,21465,1,73.5$,"08:47:31 P.M.,10/07/24",NULL,2024-10-07
1,82023,53482,21374,1,$25.00,2024-09-11T05:27:49,NULL,2024-09-11
2,81439,55123,21408,1,USD22.50,"2024-08-06,06:53",,2024-08-06
3,80259,54387,21013,1,42.5$,03/18/25;09:32:41 AM,,2025-03-18
4,80176,54916,21437,1,$ 62.5,"04:55:14 pm,09/24/24",NULL,2024-09-24


In [25]:
import re

def extract_price(val):
    if pd.isnull(val):
        return None, None
    val = str(val)
    if '$' in val or 'USD' in val:
        currency = 'USD'
    elif '€' in val or 'EUR' in val:
        currency = 'EUR'
    else:
        currency = 'UNKNOWN'
    cents_match = re.search(r'(\d+)\$(\d+)¢', val)
    if cents_match:
        amount = float(cents_match.group(1)) + float(cents_match.group(2)) / 100
        return amount, 'USD'
    match = re.search(r'[\d]+\.?\d*', val)
    if match:
        return float(match.group()), currency
    return None, currency

orders_df[['price_clean', 'currency']] = orders_df['unit_price'].apply(
    lambda x: pd.Series(extract_price(x))
)

print(orders_df[['unit_price', 'price_clean', 'currency']].head(5))
print(orders_df['currency'].value_counts())

  unit_price  price_clean currency
0      73.5$         73.5      USD
1     $25.00         25.0      USD
2   USD22.50         22.5      USD
3      42.5$         42.5      USD
4     $ 62.5         62.5      USD
currency
USD    6721
EUR    3129
Name: count, dtype: int64


In [26]:
mask = orders_df['price_clean'].isnull() | (orders_df['currency'] == 'UNKNOWN')
orders_df[mask]['unit_price'].head(20)

Series([], Name: unit_price, dtype: object)

In [27]:
rate = 1.2

orders_df['price_usd'] = orders_df.apply(
    lambda row: round(row['price_clean'] * rate, 2)
    if row['currency'] == 'EUR'
    else round(row['price_clean'], 2),
    axis=1
)

In [28]:
orders_df['paid_price'] = orders_df['quantity'] * orders_df['price_usd']
orders_df[['quantity','unit_price', 'price_clean', 'currency', 'price_usd','paid_price']].head(10)

,quantity,unit_price,price_clean,currency,price_usd,paid_price
0,1,73.5$,73.50,USD,73.50,73.50
1,1,$25.00,25.00,USD,25.00,25.00
2,1,USD22.50,22.50,USD,22.50,22.50
3,1,42.5$,42.50,USD,42.50,42.50
4,1,$ 62.5,62.50,USD,62.50,62.50
5,4,$ 73.0,73.00,USD,73.00,292.00
6,3,$ 41.75,41.75,USD,41.75,125.25
7,1,46.0 $,46.00,USD,46.00,46.00
8,5,22.USD,22.00,USD,22.00,110.00
9,2,57.00EUR,57.00,EUR,68.40,136.80


In [29]:
orders_df['shipping'] = orders_df['shipping'].replace(r'^\s*$', np.nan, regex=True)
orders_df = orders_df.replace({None: np.nan})
orders_df['shipping'].isna().sum()

np.int64(4841)

In [30]:
orders_df = orders_df.drop(columns=['unit_price','timestamp'])

In [31]:
orders_df.head()

,id,user_id,book_id,quantity,shipping,date,price_clean,currency,price_usd,paid_price
0,75198,55590,21465,1,NULL,2024-10-07,73.5,USD,73.5,73.5
1,82023,53482,21374,1,NULL,2024-09-11,25.0,USD,25.0,25.0
2,81439,55123,21408,1,NaN,2024-08-06,22.5,USD,22.5,22.5
3,80259,54387,21013,1,NaN,2025-03-18,42.5,USD,42.5,42.5
4,80176,54916,21437,1,NULL,2024-09-24,62.5,USD,62.5,62.5


In [32]:
orders_df['shipping'] = orders_df['shipping'].replace("NULL", np.nan)
orders_df['shipping'].eq("NULL").sum()

np.int64(0)

In [33]:
orders_df.head()

,id,user_id,book_id,quantity,shipping,date,price_clean,currency,price_usd,paid_price
0,75198,55590,21465,1,NaN,2024-10-07,73.5,USD,73.5,73.5
1,82023,53482,21374,1,NaN,2024-09-11,25.0,USD,25.0,25.0
2,81439,55123,21408,1,NaN,2024-08-06,22.5,USD,22.5,22.5
3,80259,54387,21013,1,NaN,2025-03-18,42.5,USD,42.5,42.5
4,80176,54916,21437,1,NaN,2024-09-24,62.5,USD,62.5,62.5


In [34]:
users_df.head()

,id,name,address,phone,email
0,53386,Jacqulyn Mante,"566 Emelina Turnpike, Lashundaside, NE 78034-5281",9627158009,leonardo.leannon@wintheiser-lueilwitz.example
1,54635,Devon Leannon III,NaN,5371339394,jon_torp@bayer.example
2,55435,Shiloh Keebler,"425 Wesley Hills, Nathanialburgh, MS 08004",4742423397,murray_mcclure@pfeffer.example
3,53627,Luke Rolfson,"Suite 556 20616 Little Union, Connieside, VT 3...",9273818818,murray.shanahan@heathcote.example
4,55650,Prince Braun,"981 Howell Spring, New Sheldon, LA 62246-1232",6861861306,anitra@pollich-kris.example


In [35]:
books_df.head()

,id,title,author,genre,publisher,year
0,21326,Behold the Man,"Trenton Sipes, Clint Hauck VM",Reference book,Bowes & Bowes,1979
1,21546,The Deer Hunter,Norris Gusikowski,Reference book,Koren Publishers Jerusalem,1993
2,21130,Snatch,Rev. Lura Jaskolski,Fable,Left Book Club,2000
3,21287,Dial M for Murder,Miss Elmo Walsh,Textbook,Libertas Academica,1977
4,21237,The Departed,Mario Aufderhar,Folklore,BBC Books,2015


In [36]:
(users_df == 'NULL').any()

id         False
name       False
address    False
phone      False
email      False
dtype: bool

In [37]:
(books_df=='NULL').any()

id           False
title        False
author       False
genre        False
publisher     True
year         False
dtype: boolean

In [38]:
books_df['publisher'] = books_df['publisher'].replace("NULL", np.nan)

In [39]:
(books_df=='NULL').any()

id           False
title        False
author       False
genre        False
publisher    False
year         False
dtype: boolean

In [40]:
(orders_df=='NULL').any()

id             False
user_id        False
book_id        False
quantity       False
shipping       False
date           False
price_clean    False
currency       False
price_usd      False
paid_price     False
dtype: bool

In [41]:
books_df.to_csv('../data2/books_clean.csv', index=False)
users_df.to_csv('../data2/users_clean.csv', index=False)
orders_df.to_csv('../data2/orders_clean.csv', index=False)